In [ ]:
# Ozone
# Data import cell (written with help from Google Gemini)

import gzip
import shutil

# Define the paths for the gzipped and uncompressed files
gzipped_file_path = '/content/2020_ozone_daily_8hour_maximum.txt.gz' # Downloaded from https://www.epa.gov/hesc/rsig-related-downloadable-data-files
uncompressed_file_path = '2020_ozone_daily_8hour_maximum.txt'

try:
    # Open the gzipped file in binary read mode ('rb')
    with gzip.open(gzipped_file_path, 'rb') as f_in:
        # Open the output file in binary write mode ('wb')
        with open(uncompressed_file_path, 'wb') as f_out:
            # Copy the decompressed content from the input to the output file
            shutil.copyfileobj(f_in, f_out)
    print(f"Successfully unzipped '{gzipped_file_path}' to '{uncompressed_file_path}'")
except FileNotFoundError:
    print(f"Error: The file '{gzipped_file_path}' was not found.")
except Exception as e:
    print(f"An error occurred during decompression: {e}")

Successfully unzipped '/content/2020_ozone_daily_8hour_maximum.txt.gz' to '2020_ozone_daily_8hour_maximum.txt'


In [ ]:
import pandas
data = pandas.read_csv("2020_ozone_daily_8hour_maximum.txt", dtype={"FIPS":str}) # Read FIPS as a string to preserve leading zeros
data

,Date,FIPS,Longitude,Latitude,ozone_daily_8hour_maximum(ppb),ozone_daily_8hour_maximum_stderr(ppb)
0,2020/01/01,01001020100,-86.49007,32.47718,36.189,4.8896
1,2020/01/01,01001020200,-86.47337,32.47434,36.310,4.8311
2,2020/01/01,01001020300,-86.46019,32.47543,36.366,4.7910
3,2020/01/01,01001020400,-86.44363,32.47200,36.386,4.8825
4,2020/01/01,01001020501,-86.42256,32.44786,36.252,4.8625
...,...,...,...,...,...,...
30662011,2020/12/31,56043000200,-107.68105,43.90472,31.599,4.9989
30662012,2020/12/31,56043000301,-107.95638,44.01437,29.632,4.6454
30662013,2020/12/31,56043000302,-107.94963,44.02720,29.562,4.6069
30662014,2020/12/31,56045951100,-104.57354,43.83987,37.102,5.1476


In [ ]:
# A rough check to make sure we've loaded all the data by looking at how many unique dates there are. 2020 was a leap year
print(len(data["Date"].unique()))
# Another check: if there are 85,528 Census Tracts (https://www.census.gov/geographies/reference-files/time-series/geo/tallies.html) and 366 days in 2020, we should have 31,303,248 observations/rows (assuming all Tracts present and with observations every day). We should at least be close....we seem to be missing ~700,000 observations.
print(len(data))
85528*366

366
30662016


31303248

In [ ]:
"""
Calculation - "The ozone indicator in EJScreen v2.3 reflects the annual mean of the 10 highest MDA8 O3 concentrations...While the form of the O3 NAAQS is based on the annual 4th highest MDA8 O3 value, here we look at an average
across the top 10 days which will span days above and below the value of the 4th high. By looking at an average across multiple days rather than a single day, this metric provides more year-to-year stability while still
representing concentrations that correspond to peak ozone exposure."
"""

# So find ten highest concentrations for each FIPS and take average
grouped = data.groupby('FIPS')[['ozone_daily_8hour_maximum(ppb)']].agg(lambda grp: grp.nlargest(10).mean()) # https://stackoverflow.com/questions/59067194/phyton-how-to-get-the-average-of-the-n-largest-values-for-each-column-grouped-b
# What the above should be doing is finding the mean of the 10 largest MDA8 O3 concentrations for each FIPS code (Census Tract).
grouped

,ozone_daily_8hour_maximum(ppb)
FIPS,
01001020100,50.6387
01001020200,50.6511
01001020300,50.7697
01001020400,50.8214
01001020501,50.8005
...,...
56043000200,60.9033
56043000301,60.0768
56043000302,60.0688


In [ ]:
grouped.describe() # In EJScreen, the raw values for the ozone indicator range from 6.95112 to 77.5368. <- according to the technical documentation (but it is outdated)
# Example FIPS to compare to EJSCREEN 01001020100 50.6387

,ozone_daily_8hour_maximum(ppb)
count,83776.000000
mean,61.886241
std,9.045459
min,38.778400
25%,55.504375
50%,60.192800
75%,66.116850
max,112.812600


In [ ]:
# Confirmation from the change log that we are dealing with 2020: "The ozone data was changed to the annual mean of the top 10 of daily maximum 8-hour concentrations. Prior to that it was incorrectly using a summer seasonal average of daily maximum 8-hour concentration. Both datasets are from 2020."
# The above change was logged in August 2024, so may not be reflected in the tech doc. Instead, need to check EJSCREEN itself.
# Validate against EJSCREEN
import geopandas
ejscreen = geopandas.read_file("/content/drive/Shareddrives/EDGI - Shared NEW/12_Environmental_Enforcement_Watch/09_Data/EJScreen_2024_BG_with_AS_CNMI_GU_VI.gdb", columns=["ID"]) # Connecting to this copy of EJSCREEN requires access to EDGI's shared drive
ejscreen

/usr/local/lib/python3.12/dist-packages/pyogrio/geopandas.py:382: UserWarning: More than one layer found in 'EJScreen_2024_BG_with_AS_CNMI_GU_VI.gdb': 'EJSCREEN_Full_with_AS_CNMI_GU_VI' (default), 'USA'. Specify layer parameter to avoid this warning.
  result = read_func(


,ID,STATE_NAME,ST_ABBREV,CNTY_NAME,REGION,ACSTOTPOP,ACSIPOVBAS,ACSEDUCBAS,ACSTOTHH,ACSTOTHU,...,AREAWATER,NPL_CNT,TSDF_CNT,EXCEED_COUNT_80,EXCEED_COUNT_80_SUP,DEMOGIDX_2ST,DEMOGIDX_5ST,Shape_Length,Shape_Area,geometry
0,010010201001,Alabama,AL,Autauga County,4,558.0,558.0,423.0,261.0,279.0,...,28435.0,0.0,0.0,1.0,4.0,1.362959,2.249444,0.110793,0.000412,"MULTIPOLYGON (((-86.51038 32.47225, -86.5103 3..."
1,010010201002,Alabama,AL,Autauga County,4,1307.0,1307.0,843.0,439.0,454.0,...,0.0,0.0,0.0,1.0,2.0,0.796153,1.929149,0.096868,0.000534,"MULTIPOLYGON (((-86.50461 32.47723, -86.50453 ..."
2,010010202001,Alabama,AL,Autauga County,4,548.0,540.0,428.0,204.0,276.0,...,0.0,0.0,0.0,3.0,2.0,1.840541,1.687681,0.062129,0.000197,"MULTIPOLYGON (((-86.48127 32.47744, -86.48126 ..."
3,010010202002,Alabama,AL,Autauga County,4,1313.0,1129.0,1047.0,340.0,404.0,...,5669.0,0.0,0.0,1.0,2.0,1.529810,1.675390,0.052097,0.000122,"MULTIPOLYGON (((-86.47611 32.46765, -86.47564 ..."
4,010010203001,Alabama,AL,Autauga County,4,2835.0,2829.0,1863.0,1011.0,1115.0,...,9054.0,0.0,0.0,1.0,1.0,0.958818,1.511979,0.089441,0.000372,"MULTIPOLYGON (((-86.47087 32.47573, -86.47084 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243017,7801035150,Virgin Islands,VI,St. Croix Island,2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.053795,0.000125,"MULTIPOLYGON (((-64.83365 17.69202, -64.83084 ..."
243018,7801007850,Virgin Islands,VI,St. Croix Island,2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065407,0.000157,"MULTIPOLYGON (((-64.77104 17.72041, -64.76549 ..."
243019,7803054875,Virgin Islands,VI,St. Thomas Island,2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.092625,0.000252,"MULTIPOLYGON (((-64.98406 18.3377, -64.9759 18..."
243020,7802008700,Virgin Islands,VI,St. John Island,2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.050068,0.000096,"MULTIPOLYGON (((-64.71227 18.36273, -64.71334 ..."


In [ ]:
# Check min/max on "raw" ozone score in EJSCREEN
# Median and distribution expected to be somewhat different due to the fact that in the above use of the O3 data we have not splayed the Tract values to Block Groups
ejscreen["OZONE"].describe()

,OZONE
count,238194.000000
mean,61.849590
std,8.966176
min,38.778420
25%,55.515330
50%,60.254170
75%,66.039515
max,112.812760
